# 03 · Agentes paso a paso

Un agente es un modelo de lenguaje al que se le dan herramientas: funciones que puede pedir que se ejecuten. El modelo no ejecuta nada; solo dice qué herramienta quiere y con qué argumentos. Nuestro programa la ejecuta, le devuelve el resultado y el modelo sigue.

Aquí se construye ese ciclo a mano, sin ningún framework. Es el mismo ciclo que dibuja la pestaña **Agentes** de la interfaz web.

1. Una herramienta: consultar la base de datos de producción.
2. El ciclo, un paso a la vez.
3. El ciclo completo en una función.
4. Un agente que le delega a otro.

In [ ]:
import json
import os
import sqlite3
import time
from pathlib import Path

from openai import OpenAI

MODEL = os.environ["LLM_MODEL"]
client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ.get("LLM_API_KEY", "local"))

print("Modelo:", MODEL)

## 1. Una herramienta

Los datos son de ejemplo: producción, scrap y paros de tres líneas durante los últimos 14 días. Los genera el backend del taller cada vez que arranca.

In [ ]:
DB_PATH = Path("/workspace/data/produccion.sqlite")

if not DB_PATH.exists():
    raise FileNotFoundError(
        "No está la base de datos de ejemplo. Arranca el backend y vuelve a ejecutar: docker compose up -d backend"
    )

connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
TABLE_SQL = connection.execute("SELECT sql FROM sqlite_master WHERE name = 'produccion'").fetchone()[0]
LAST_DAY = connection.execute("SELECT MAX(fecha) FROM produccion").fetchone()[0]
connection.close()

print(TABLE_SQL)
print("Último día con datos:", LAST_DAY)

La herramienta es una función normal de Python. Solo lee: abre la base en modo de solo lectura y rechaza todo lo que no sea un `SELECT`.

In [ ]:
def query_production(sql):
    if not sql.lstrip().lower().startswith("select"):
        raise ValueError("Solo se permiten consultas SELECT.")
    connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    try:
        cursor = connection.execute(sql)
        return {"columnas": [column[0] for column in cursor.description], "filas": cursor.fetchmany(50)}
    finally:
        connection.close()


query_production("SELECT linea, SUM(piezas_producidas) AS piezas FROM produccion GROUP BY linea")

Al modelo no se le pasa la función, sino su descripción: nombre, para qué sirve y qué argumentos recibe. Con eso decide cuándo pedirla.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "query_production",
            "description": "Ejecuta una consulta SELECT de SQLite sobre la tabla produccion y devuelve las filas.",
            "parameters": {
                "type": "object",
                "properties": {"sql": {"type": "string", "description": "Consulta SELECT de SQLite."}},
                "required": ["sql"],
            },
        },
    }
]

# Qué función de Python corresponde a cada nombre de herramienta.
FUNCTIONS = {"query_production": query_production}

ANALYST = (
    "Eres analista de datos de producción. Para responder usa siempre la herramienta query_production "
    f"con SQL de SQLite sobre esta tabla:\n{TABLE_SQL}\n"
    f"El último día con datos es {LAST_DAY}. "
    "Si te preguntan qué anda mal o qué hay que atender, compara por linea y turno el porcentaje de scrap "
    "(piezas_scrap * 100.0 / piezas_producidas) y el paro promedio de los últimos 4 días. "
    "Responde en español, en pocas líneas, solo con cifras que aparezcan en los resultados de tus consultas."
)

QUESTION = "¿Qué línea y turno tuvo el mayor porcentaje de scrap en los últimos 4 días?"

## 2. El ciclo, un paso a la vez

**Paso 1.** Le mandamos la pregunta junto con la lista de herramientas. El modelo todavía no puede responder, porque no tiene los datos: en lugar de texto devuelve una petición de herramienta.

In [ ]:
messages = [
    {"role": "system", "content": ANALYST},
    {"role": "user", "content": QUESTION},
]

response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
message = response.choices[0].message

print("Texto de la respuesta:", repr(message.content))
for call in message.tool_calls or []:
    print("Herramienta pedida:   ", call.function.name)
    print("Argumentos:           ", call.function.arguments)

**Paso 2.** Ejecutamos nosotros lo que pidió y agregamos el resultado a la conversación, como un mensaje de tipo `tool`.

In [ ]:
def assistant_message(message):
    # La petición del modelo se guarda en la conversación tal como llegó.
    entry = {"role": "assistant", "content": message.content or ""}
    if message.tool_calls:
        entry["tool_calls"] = [
            {"id": call.id, "type": "function", "function": {"name": call.function.name, "arguments": call.function.arguments}}
            for call in message.tool_calls
        ]
    return entry


def run_tool(call, functions):
    # Si la herramienta falla, el error se le devuelve al modelo para que lo corrija.
    try:
        return functions[call.function.name](**json.loads(call.function.arguments or "{}"))
    except Exception as error:
        return {"error": f"{type(error).__name__}: {error}"}


messages.append(assistant_message(message))

for call in message.tool_calls or []:
    result = run_tool(call, FUNCTIONS)
    print(json.dumps(result, ensure_ascii=False, indent=1)[:600])
    messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result, ensure_ascii=False)})

if not message.tool_calls:
    print("Esta vez el modelo respondió sin pedir la herramienta. Vuelve a ejecutar desde el paso 1.")

**Paso 3.** Volvemos a llamar al modelo con la conversación completa. Ahora ya tiene los datos y puede responder, o pedir otra consulta.

In [ ]:
response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
message = response.choices[0].message

if message.tool_calls:
    print("Pide otra consulta:", message.tool_calls[0].function.arguments)
else:
    print(message.content)

## 3. El ciclo completo

Eso es todo un agente: repetir los pasos 1 a 3 hasta que el modelo responda con texto. La función imprime cada paso, que es justo lo que la interfaz web convierte en nodos que se encienden y en la lista "Paso a paso".

In [ ]:
MAX_STEPS = 6


def run_agent(name, system, task, tools, functions, depth=0):
    pad = "    " * depth
    print(f"{pad}[{name}] empieza: {task}")
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]

    for step in range(MAX_STEPS + 1):
        # En el último intento ya no se le ofrecen herramientas: tiene que responder.
        offered = tools if step < MAX_STEPS else None
        response = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0, **({"tools": offered} if offered else {})
        )
        message = response.choices[0].message

        if not message.tool_calls:
            print(f"{pad}[{name}] responde ({response.usage.completion_tokens} tokens)")
            return message.content or ""

        messages.append(assistant_message(message))
        for call in message.tool_calls:
            print(f"{pad}  -> {call.function.name} {call.function.arguments}")
            start = time.perf_counter()
            if call.function.name in functions:
                result = run_tool(call, functions)
            else:
                result = {"error": f"La herramienta '{call.function.name}' no existe."}
            if not isinstance(result, str):
                result = json.dumps(result, ensure_ascii=False)
            print(f"{pad}  <- {result[:160]}{'...' if len(result) > 160 else ''} ({(time.perf_counter() - start) * 1000:.0f} ms)")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return ""


answer = run_agent("analista", ANALYST, QUESTION, TOOLS, FUNCTIONS)
print()
print(answer)

## 4. Un agente que le delega a otro

Delegar no necesita nada nuevo: para el supervisor, el analista es una herramienta más. Cuando la pide, nuestro programa corre el ciclo del analista y le devuelve su respuesta como resultado.

In [ ]:
def ask_analyst(task):
    return run_agent("analista", ANALYST, task, TOOLS, FUNCTIONS, depth=1)


SUPERVISOR = (
    "Eres el supervisor de turno de una planta. No tienes los datos de producción: para cualquier cifra "
    "pídesela al analista con la herramienta ask_analyst. Con lo que te responda, dile al usuario en dos o tres "
    "líneas qué debe atender. No inventes cifras."
)

SUPERVISOR_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "ask_analyst",
            "description": "Delega una tarea al analista de datos, que consulta producción, scrap y paros por fecha, turno y línea.",
            "parameters": {
                "type": "object",
                "properties": {"task": {"type": "string", "description": "Qué debe averiguar, con todo el contexto."}},
                "required": ["task"],
            },
        },
    }
]

answer = run_agent(
    "supervisor",
    SUPERVISOR,
    "¿Hay alguna línea o turno que deba atender antes del siguiente turno?",
    SUPERVISOR_TOOLS,
    {"ask_analyst": ask_analyst},
)
print()
print(answer)

## Del notebook a la interfaz

El backend del taller hace lo mismo que `run_agent`, con dos diferencias: en lugar de imprimir cada paso publica un evento, y la interfaz dibuja esos eventos conforme llegan.

| Aquí se imprime | Evento | En la pantalla |
|---|---|---|
| `[agente] empieza` | `agent_start` | se enciende el nodo del agente |
| `-> herramienta {...}` | `tool_call` | se enciende la herramienta y se anima la flecha |
| `<- resultado` | `tool_result` | el resultado, con su duración |
| `-> ask_analyst` | `handoff` | la flecha de un agente a otro |
| `[agente] responde` | `agent_end` | el nodo queda marcado como terminado |

Un modelo pequeño se equivoca más: a veces responde sin consultar, o consulta una sola vez y completa el resto por su cuenta. Vale la pena repetir este notebook con un modelo más grande cuando termine de descargarse (cambia `MODEL` en la primera celda) y comparar las consultas que hace cada uno.